# SigLIP2 200K inference and API flow

This notebook is a runnable bridge between the completed 200K experiment and a production API. It shows how to load the exact promoted checkpoint, reproduce catalog text preprocessing, create image and text embeddings for unseen inputs, build a small in-memory retrieval index from the saved feature shards, and run top-k searches.

The deployed model is a composition, not one standalone Hugging Face folder:

`input -> versioned preprocessing -> frozen google/siglip2-base-patch16-224 -> trained 200K residual adapter -> normalized 768-dimensional embedding -> vector index`

The notebook defaults to 20,000 catalog products so it remains comfortable on the Mac. Set `CATALOG_LIMIT = None` to use all 198,130 successfully embedded pilot products.

## What was saved

| Artifact | Role | Git status |
|---|---|---|
| `checkpoints/pilot_200k/taxonomy_w20_random_200k.pt` | Learned image adapter, text adapter, logit parameters, training config, and feature metadata | Versioned in Git as the promoted compact deployment adapter |
| `features/pilot_200k/` | Frozen base embeddings and product IDs for 198,130 products | Local only; ignored by Git |
| `artifacts/pilot_200k.jsonl.gz` | Processed 200K metadata sample with cleaned/chunked descriptions | Local only; ignored by Git |
| `checkpoints/pilot_200k/results.json` and `manifest.json` | Metrics, selection decision, and lineage | Versioned with the promoted checkpoint |
| `configs/`, `src/`, `tests/`, and `docs/` | Reproducible methodology and implementation | Intended for Git |

The checkpoint is about 1.5 MB because the frozen base model is not duplicated. In deployment, retain the checkpoint, the exact base-model revision/cache, `description_v1.yaml`, and an exported vector index.

In [ ]:
from pathlib import Path
import json
import sys

cwd = Path.cwd().resolve()
if (cwd / 'src' / 'siglip2_training').exists():
    PROJECT = cwd
elif (cwd / 'SigLIP2_training' / 'src' / 'siglip2_training').exists():
    PROJECT = cwd / 'SigLIP2_training'
elif cwd.name == 'notebooks' and (cwd.parent / 'src' / 'siglip2_training').exists():
    PROJECT = cwd.parent
else:
    raise FileNotFoundError('Run this notebook from the repository root or SigLIP2_training directory.')

sys.path.insert(0, str(PROJECT / 'src'))
CHECKPOINT_PATH = PROJECT / 'checkpoints/pilot_200k/taxonomy_w20_random_200k.pt'
FEATURE_DIR = PROJECT / 'features/pilot_200k'
RESULTS_PATH = PROJECT / 'checkpoints/pilot_200k/results.json'
DESCRIPTION_CONFIG_PATH = PROJECT / 'configs/description_v1.yaml'
FEATURE_CONFIG_PATH = PROJECT / 'configs/features_50k.yaml'

for path in [CHECKPOINT_PATH, RESULTS_PATH, DESCRIPTION_CONFIG_PATH]:
    assert path.exists(), f'Missing required artifact: {path}'
FEATURES_AVAILABLE = (FEATURE_DIR / 'index.json').exists()

results = json.loads(RESULTS_PATH.read_text())
print('Project:', PROJECT)
print('Selected model:', results['selection']['candidate'])
print('Train/validation/test:', results['rows'])
print('Held-out bidirectional Recall@10:', f"{results['held_out_test']['selected_retrieval']['bidirectional_recall@10']:.2%}")
print('Local catalog feature index available:', FEATURES_AVAILABLE)

## 1. Load the frozen encoder and trained adapters

The SigLIP2 backbone converts images or text into base 768-dimensional vectors. Training kept that backbone frozen and learned two small residual transformations: one for image vectors and one for text vectors. Both sides therefore continue to learn a shared retrieval space through the adapters.

`LOCAL_FILES_ONLY=True` reproduces the training environment. Set it to `False` once on a new machine if the base model has not yet been downloaded. For a production image, pin and bake the model files rather than downloading at request time.

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoImageProcessor, AutoModel, AutoTokenizer

from siglip2_training.config import load_yaml
from siglip2_training.features import choose_device
from siglip2_training.training import _torch_modules, text_view

DEVICE = choose_device(['cuda', 'mps', 'cpu'])
LOCAL_FILES_ONLY = True
feature_config = load_yaml(FEATURE_CONFIG_PATH)
model_id = feature_config['model']['id']

checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
base_model = AutoModel.from_pretrained(model_id, local_files_only=LOCAL_FILES_ONLY).to(DEVICE).eval()
tokenizer = AutoTokenizer.from_pretrained(model_id, local_files_only=LOCAL_FILES_ONLY, use_fast=True)
image_processor = AutoImageProcessor.from_pretrained(model_id, local_files_only=LOCAL_FILES_ONLY, use_fast=False)

_, ContrastiveAdapters = _torch_modules()
adapter_cfg = checkpoint['training_config']['adapter']
feature_index = checkpoint['feature_index']
adapter = ContrastiveAdapters(
    dimension=int(checkpoint['dimension']),
    bottleneck=int(adapter_cfg['bottleneck_dimension']),
    dropout=float(adapter_cfg['dropout']),
    logit_scale=float(feature_index['pretrained_logit_scale']),
    logit_bias=float(feature_index['pretrained_logit_bias']),
    maximum_logit_scale=float(adapter_cfg['maximum_logit_scale']),
).to(DEVICE)
adapter.load_state_dict(checkpoint['state_dict'])
adapter.eval()

print({'device': DEVICE, 'base_model': model_id, 'dimension': checkpoint['dimension'], 'strategy': checkpoint['strategy']})

## 2. Use the exact catalog-description workflow

Catalog products and user search queries intentionally follow different paths:

- **Catalog product:** normalize title/description/features, remove duplicate and boilerplate sentences, produce up to four overlapping 64-token chunks, mean-pool chunk embeddings, blend 20% taxonomy when taxonomy exists, then apply the text adapter.
- **Natural-language query:** normalize whitespace/Unicode, encode the query once with the 64-token limit, then apply the text adapter. We do not invent taxonomy or remove product-marketing phrases from a user's query.

This distinction should remain identical in the API.

In [ ]:
from dataclasses import asdict
from siglip2_training.schema import ProductInput
from siglip2_training.text import DescriptionProcessor

description_config = load_yaml(DESCRIPTION_CONFIG_PATH)
description_processor = DescriptionProcessor(description_config, tokenizer=tokenizer)
MAX_TOKENS = int(description_config['chunking']['maximum_tokens'])
TAXONOMY_WEIGHT = float(checkpoint['candidate']['taxonomy_weight'])

def encode_base_texts(texts):
    inputs = tokenizer(
        list(texts), padding=True, truncation=True, max_length=MAX_TOKENS, return_tensors='pt'
    )
    inputs = {key: value.to(DEVICE) for key, value in inputs.items()}
    with torch.inference_mode():
        return F.normalize(base_model.get_text_features(**inputs), dim=-1)

def adapt_text(base_embeddings):
    with torch.inference_mode():
        return adapter.text_adapter(base_embeddings.to(DEVICE))

def embed_search_query(query):
    normalized = description_processor.normalize(query)
    return adapt_text(encode_base_texts([normalized]))[0].cpu().numpy()

def embed_catalog_product(product):
    transformed = description_processor.transform(product)
    if not transformed.description_chunks:
        raise ValueError('Product has no usable description chunks.')
    chunk_vectors = encode_base_texts(transformed.description_chunks)
    pooled = F.normalize(chunk_vectors.mean(dim=0, keepdim=True), dim=-1)
    if transformed.taxonomy_text:
        taxonomy = encode_base_texts([transformed.taxonomy_text])
        pooled = F.normalize((1 - TAXONOMY_WEIGHT) * pooled + TAXONOMY_WEIGHT * taxonomy, dim=-1)
    return transformed, adapt_text(pooled)[0].cpu().numpy()

example_product = ProductInput(
    product_id='api-demo-1',
    title='Modern walnut coffee table with storage shelf',
    descriptions=('Rectangular living-room table with a lower shelf and durable wood finish.',),
    features=('Open storage shelf', 'Easy assembly'),
    details={'Material': 'Walnut wood', 'Color': 'Brown', 'Product Dimensions': '42 x 22 x 18 inches'},
    categories=('Home & Kitchen', 'Furniture', 'Living Room Furniture', 'Coffee Tables'),
)
transformed, product_text_embedding = embed_catalog_product(example_product)
print('Taxonomy:', transformed.taxonomy_text)
print('Chunks:', len(transformed.description_chunks))
print('Embedding shape:', product_text_embedding.shape)

## 3. Embed an unseen image

The API may accept uploaded bytes, a server-local path, or a URL fetched by a controlled ingestion worker. Production URL fetching needs timeouts, content-length limits, MIME validation, and SSRF protection; the compact helper below is for local experimentation.

In [ ]:
from io import BytesIO
from PIL import Image
import requests

def load_rgb_image(source):
    if isinstance(source, Image.Image):
        return source.convert('RGB')
    value = str(source)
    if value.startswith(('http://', 'https://')):
        response = requests.get(value, timeout=20)
        response.raise_for_status()
        return Image.open(BytesIO(response.content)).convert('RGB')
    return Image.open(value).convert('RGB')

def embed_image(source):
    image = load_rgb_image(source)
    inputs = image_processor(images=[image], return_tensors='pt')
    inputs = {key: value.to(DEVICE) for key, value in inputs.items()}
    with torch.inference_mode():
        base = F.normalize(base_model.get_image_features(**inputs), dim=-1)
        final = adapter.image_adapter(base)
    return final[0].cpu().numpy()

# Example: image_embedding = embed_image('/absolute/path/to/product.jpg')

## 4. Build the placeholder retrieval index

The saved feature shards contain frozen base embeddings. This cell streams the shards, applies the selected adapters, and returns aligned image and description indexes. For an API, run this once offline and save the final vectors in FAISS, Qdrant, pgvector, Pinecone, or another vector store. Do not rebuild the index during web-server startup.

In [ ]:
import pandas as pd

CATALOG_LIMIT = 20_000  # Set to None for all 198,130 embedded pilot products.
ADAPTER_BATCH_SIZE = 2_048

def adapt_numpy(values, side):
    output = []
    layer = adapter.image_adapter if side == 'image' else adapter.text_adapter
    for start in range(0, len(values), ADAPTER_BATCH_SIZE):
        batch = torch.from_numpy(values[start:start + ADAPTER_BATCH_SIZE].astype(np.float32)).to(DEVICE)
        with torch.inference_mode():
            output.append(layer(batch).cpu().numpy().astype(np.float32))
    return np.concatenate(output)

def load_catalog_indexes(limit=CATALOG_LIMIT):
    if not FEATURES_AVAILABLE:
        raise FileNotFoundError(
            'Catalog feature shards are external artifacts. Mount features/pilot_200k or connect a vector database.'
        )
    index_meta = json.loads((FEATURE_DIR / 'index.json').read_text())
    metadata_parts, image_parts, text_parts = [], [], []
    remaining = limit
    for shard_name in index_meta['shards']:
        with np.load(FEATURE_DIR / shard_name, allow_pickle=False) as shard_file:
            take = len(shard_file['product_id']) if remaining is None else min(remaining, len(shard_file['product_id']))
            if take <= 0:
                break
            shard = {key: shard_file[key][:take] for key in shard_file.files}
            base_text = text_view(shard, 'multichunk_with_taxonomy', TAXONOMY_WEIGHT)
            image_parts.append(adapt_numpy(shard['image'], 'image'))
            text_parts.append(adapt_numpy(base_text, 'text'))
            metadata_parts.append(pd.DataFrame({
                'product_id': shard['product_id'].astype(str),
                'category_path': shard['category_path'].astype(str),
                'leaf_category': shard['leaf_category'].astype(str),
                'split': shard['split'].astype(str),
            }))
            if remaining is not None:
                remaining -= take
    metadata = pd.concat(metadata_parts, ignore_index=True)
    return metadata, np.vstack(image_parts), np.vstack(text_parts)

catalog, catalog_image_vectors, catalog_text_vectors = load_catalog_indexes()
print('Indexed products:', f'{len(catalog):,}')
print('Image index:', catalog_image_vectors.shape, 'Text index:', catalog_text_vectors.shape)

## 5. Run the two API-style retrieval workflows

- **Visual substitutes:** embed an image with the image tower and search the catalog image index.
- **Text search:** embed a user query with the text tower and search the catalog image index. This is cross-modal retrieval and returns products whose images align with the query.
- **Description similarity (optional):** search the catalog text index instead of the image index.

In [ ]:
def top_k(query_vector, index_vectors, k=10, exclude_product_id=None):
    scores = index_vectors @ query_vector.astype(np.float32)
    if exclude_product_id is not None:
        scores[catalog['product_id'].to_numpy() == str(exclude_product_id)] = -np.inf
    k = min(k, len(scores))
    positions = np.argpartition(-scores, k - 1)[:k]
    positions = positions[np.argsort(-scores[positions])]
    result = catalog.iloc[positions].copy()
    result.insert(1, 'score', scores[positions])
    return result.reset_index(drop=True)

query = 'minimalist walnut coffee table with storage for a modern living room'
query_vector = embed_search_query(query)
print('Cross-modal text -> catalog image results')
display(top_k(query_vector, catalog_image_vectors, k=10))

print('Text -> catalog description results')
display(top_k(query_vector, catalog_text_vectors, k=10))

In [ ]:
# A no-download visual-similarity example using an existing indexed product.
query_row = 0
query_product_id = catalog.iloc[query_row]['product_id']
display(top_k(
    catalog_image_vectors[query_row],
    catalog_image_vectors,
    k=10,
    exclude_product_id=query_product_id,
))

# For an unseen image instead:
# unseen_vector = embed_image('/absolute/path/to/new_product.jpg')
# display(top_k(unseen_vector, catalog_image_vectors, k=10))

## 6. True unseen-image API demo

This demo uses `B004K6GWSE.jpg`, a nightstand image stored in the earlier furniture sample. The cell first proves that its product ID does **not** occur anywhere in the complete 198,130-product pilot feature set. It then sends only the image bytes through the same interface an upload endpoint would use. No description, category, or product ID is given to the model.

The request flow is:

```text
POST image bytes
  -> request-size and image validation
  -> RGB decoding
  -> SigLIP2 image processor
  -> frozen SigLIP2 image encoder
  -> trained 200K image adapter
  -> normalized 768-dimensional vector
  -> cosine search against catalog image vectors
  -> JSON-ready top-10 products
```

The product-ID membership check below is only experimental proof. A real API receives anonymous bytes and does not perform that check.

In [ ]:
from PIL import UnidentifiedImageError

MAX_UPLOAD_BYTES = 10 * 1024 * 1024
UNSEEN_IMAGE_PATH = PROJECT.parent / 'data/images_furniture/B004K6GWSE.jpg'
EXPECTED_PRODUCT_TYPE = 'Nightstand'  # For human evaluation only; never sent to the model.

def load_all_pilot_product_ids():
    index_meta = json.loads((FEATURE_DIR / 'index.json').read_text())
    product_ids = set()
    for shard_name in index_meta['shards']:
        with np.load(FEATURE_DIR / shard_name, allow_pickle=False) as shard:
            product_ids.update(shard['product_id'].astype(str))
    return product_ids

pilot_product_ids = load_all_pilot_product_ids()
unseen_product_id = UNSEEN_IMAGE_PATH.stem
assert unseen_product_id not in pilot_product_ids, 'The selected demo product leaked into the 200K pilot.'
print({
    'image': str(UNSEEN_IMAGE_PATH),
    'product_id': unseen_product_id,
    'present_in_200k_pilot': unseen_product_id in pilot_product_ids,
    'expected_product_type_for_human_check': EXPECTED_PRODUCT_TYPE,
})
display(Image.open(UNSEEN_IMAGE_PATH).convert('RGB').resize((300, 300)))

In [ ]:
def decode_uploaded_image(payload):
    if not isinstance(payload, (bytes, bytearray)) or not payload:
        raise ValueError('The request must contain non-empty image bytes.')
    if len(payload) > MAX_UPLOAD_BYTES:
        raise ValueError(f'Image exceeds the {MAX_UPLOAD_BYTES // 1024**2} MiB request limit.')
    try:
        with Image.open(BytesIO(payload)) as candidate:
            candidate.verify()
        with Image.open(BytesIO(payload)) as candidate:
            return candidate.convert('RGB')
    except (UnidentifiedImageError, OSError) as error:
        raise ValueError('Uploaded bytes are not a supported, valid image.') from error

def search_image_upload(payload, filename='upload.jpg', k=10):
    image = decode_uploaded_image(payload)
    query_vector = embed_image(image)
    matches = top_k(query_vector, catalog_image_vectors, k=k)
    return {
        'model': 'siglip2-home-kitchen-200k',
        'embedding_dimension': int(query_vector.shape[0]),
        'query': {'filename': filename, 'bytes': len(payload)},
        'results': [
            {
                'rank': rank,
                'product_id': str(row.product_id),
                'score': round(float(row.score), 6),
                'category_path': str(row.category_path),
                'leaf_category': str(row.leaf_category),
            }
            for rank, row in enumerate(matches.itertuples(index=False), start=1)
        ],
    }

# Simulate `await UploadFile.read()` from an API request.
uploaded_bytes = UNSEEN_IMAGE_PATH.read_bytes()
api_response = search_image_upload(uploaded_bytes, filename=UNSEEN_IMAGE_PATH.name, k=10)
print(json.dumps({key: value for key, value in api_response.items() if key != 'results'}, indent=2))
display(pd.DataFrame(api_response['results']))

A minimal FastAPI route would pass request bytes into the same service method:

```python
@router.post('/v1/search/image')
async def search_image(file: UploadFile, k: int = 10):
    payload = await file.read()
    return model_service.search_image_upload(payload, filename=file.filename, k=k)
```

In production, `model_service` and the vector index are created once during application startup. The endpoint should not reload the checkpoint or rebuild the catalog index for each call.

## Files for the API implementation

A small, clean first API should contain:

```text
api/
├── app.py                 FastAPI creation, startup, health endpoint
├── settings.py            Paths, model ID/revision, device, limits
├── schemas.py             Request/response validation
├── model_service.py       Load base model + checkpoint once; embedding methods
├── preprocessing.py       Thin wrapper around the versioned DescriptionProcessor
├── vector_index.py        Load/search FAISS or vector-database index
└── routes/
    ├── embeddings.py      POST /embeddings/text and /embeddings/image
    └── search.py          POST /search/text and /search/image
scripts/
└── build_serving_index.py Apply adapters to catalog shards and export vectors/metadata
tests/
├── test_preprocessing_parity.py
├── test_embedding_contract.py
└── test_search_api.py
model_artifacts/           External/model-volume storage, not normal Git
├── taxonomy_w20_random_200k.pt
├── catalog.faiss
├── catalog_metadata.parquet
└── serving_manifest.json
```

The serving manifest should pin the checkpoint checksum, base model ID and revision, description-pipeline version/config hash, vector dimension, taxonomy weight, index metric, and catalog build timestamp.

Recommended first endpoints:

- `GET /health` — model/index readiness and artifact versions.
- `POST /v1/embeddings/text` — natural query or structured catalog product.
- `POST /v1/embeddings/image` — uploaded image; URL ingestion can come later.
- `POST /v1/search/text` — cross-modal product search.
- `POST /v1/search/image` — visual substitutes.

For the placeholder, keep one process/model worker per GPU and load model/index objects once at startup. Avoid returning raw internal paths, accepting arbitrary server-side file paths, or downloading unvalidated client URLs inside the request handler.

## Production handoff checklist

1. Copy the 200K adapter checkpoint to durable artifact storage and record its SHA-256 checksum.
2. Pin the Hugging Face base-model revision and bake it into the serving image or model volume.
3. Move the adapter classes out of the private training helper into a public inference module.
4. Build final adapted catalog vectors offline and store them in a vector index.
5. Add preprocessing-parity tests using fixed products and expected hashes.
6. Add API limits, authentication, image validation, observability, and batch embedding.
7. Swap the 200K checkpoint/index for the full-catalog versions behind the same interfaces later.